In [ ]:




# 2. SHAP 연산을 위한 가상 데이터 준비 (넘파이 어레이)
# Background: SHAP이 기여도를 계산할 때 기준으로 삼을 데이터셋 (Train 데이터 중 일부)
X_train_sample = np.random.rand(50, timesteps, input_size).astype(np.float32)
# Test: 우리가 직접 해석하고 싶은 샘플 데이터 (예: 이상치가 의심되는 시점)
X_test_sample = np.random.rand(5, timesteps, input_size).astype(np.float32)


# ⭐ [해결 핵심 1] SHAP 주입을 위해 3차원 배경 데이터를 2차원으로 펼침 (Shape: [50, 1320])
X_train_flattened = X_train_sample[:10].reshape(10, -1) 
X_test_flattened = X_test_sample[0:1].reshape(1, -1)     # (Shape: [1, 1320])


# 2. 🔥 SHAP용 래퍼 함수 수정
# SHAP이 2차원 데이터를 밀어 넣어주므로, 함수 내부에서 다시 3차원으로 복구하여 모델에 던집니다.
def model_loss_wrapper_v2(x_flattened):
    # SHAP이 보낸 2차원 데이터를 모델이 아는 3차원 [Batch, 60, 22] 구조로 원상복구
    x_3d = x_flattened.reshape(-1, timesteps, input_size)
    
    x_tensor = torch.tensor(x_3d, dtype=torch.float32)
    with torch.no_grad():
        reconstructed = model(x_tensor) # 모델 연산
        # 샘플별 전체 시점 및 변수의 평균 절대 오차(MAE) 계산
        mae_loss = torch.mean(torch.abs(x_tensor - reconstructed), dim=(1, 2))
        
    return mae_loss.cpu().numpy()


# 1. SHAP 래퍼 함수 정의 (특정 순간의 오차 매트릭을 타깃으로 지정)
def model_moment_error_wrapper(x_flattened):
    x_3d = x_flattened.reshape(-1, timesteps, input_size)
    x_tensor = torch.tensor(x_3d, dtype=torch.float32)
    
    with torch.no_grad():
        outputs = model(x_tensor)
        
        # 🎯 [핵심] 특정 순간(-1: 마지막 시점)의 실제값(x_tensor)과 예측값(outputs) 간의 오차 계산
        # 특정 순간의 변수별 절대 오차를 구한 뒤 변수 방향으로 평균을 내어 샘플별 매트릭 스칼라 생성
        moment_error = torch.mean(torch.abs(x_tensor[:, -1, :] - outputs[:, -1, :]), dim=1)
        
    return moment_error.cpu().numpy()


# 3. Explainer 선언 (2차원 데이터셋 전달)
# explainer = shap.KernelExplainer(model_loss_wrapper_v2, X_train_flattened)
explainer = shap.KernelExplainer(model_moment_error_wrapper, X_train_flattened)


# 4. SHAP Value 계산 (차원 에러 없이 깔끔하게 통과합니다)
print("SHAP 기여도 분석 연산 시작 (차원 복구 버전)...")
shap_values_flattened_1 = explainer.shap_values(X_train_flattened)
# shap_values_flattened_2 = explainer.shap_values(X_test_flattened)


# 5. ⭐ [해결 핵심 2] 계산 완료 후, 시각화를 위해 다시 3차원으로 복원
# 구조: [1, 1320] -> [60, 22] (1개 샘플 생략)
shap_values_3d_1 = shap_values_flattened_1.reshape(timesteps, input_size)
# shap_values_3d_2 = shap_values_flattened_2.reshape(timesteps, input_size)
actual_x_3d_1 = X_train_sample[0] # 대상 샘플의 원본 데이터 [60, 22]
# actual_x_3d_2 = X_test_sample[0] # 대상 샘플의 원본 데이터 [60, 22]


# 6. 시각화를 위한 22개 피처별 평균 기여도 압축 (60개 타임스텝 평균)
mean_shap_features_1 = np.mean(shap_values_3d_1, axis=0) # [22]
# mean_shap_features_2 = np.mean(shap_values_3d_2, axis=0) # [22]
mean_x_features_1 = np.mean(actual_x_3d_1, axis=0)       # [22]
# mean_x_features_2 = np.mean(actual_x_3d_2, axis=0)       # [22]

feature_names = [f"Feature_{i}" for i in range(input_size)]

# 7. 요약 그래프 출력
shap.summary_plot(
    mean_shap_features_1.reshape(1, -1), 
    mean_x_features_1.reshape(1, -1), 
    feature_names=feature_names,
    plot_type="bar"
)

# # 7. 요약 그래프 출력
# shap.summary_plot(
#     mean_shap_features_2.reshape(1, -1), 
#     mean_x_features_2.reshape(1, -1), 
#     feature_names=feature_names,
#     plot_type="bar"
# )

SHAP 기여도 분석 연산 시작 (차원 복구 버전)...


  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:15<00:00,  1.56s/it]


ValueError: cannot reshape array of size 13200 into shape (60,22)

In [6]:
import torch
import torch.nn as nn
import numpy as np
import shap

# 1. 시뮬레이션을 위한 간단한 LSTM Autoencoder 모델 정의
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, timesteps):
        super(LSTMAutoencoder, self).__init__()
        # Encoder
        self.encoder_lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        # Decoder
        self.decoder_lstm = nn.LSTM(hidden_size, input_size, batch_first=True)
        self.timesteps = timesteps
        
    def forward(self, x):
        # x Shape: [Batch, Timesteps, Features]
        _, (hidden, _) = self.encoder_lstm(x)
        # hidden의 마지막 상태를 타임스텝만큼 반복 복사하여 Decoder 입력으로 준비
        x_hat = hidden.repeat(self.timesteps, 1, 1).transpose(0, 1)
        out, _ = self.decoder_lstm(x_hat)
        return out  # 원래 x와 동일한 Shape의 복원값 반환

In [7]:
# 가상의 인프라 세팅 (CPU 환경 권장 - SHAP의 PyTorch 연산은 CPU에서 가장 안정적임)
device = torch.device('cpu')
input_size = 22     # 직전 대화에서 조율된 Feature 크기 22 설정
timesteps = 60      # Window Size 60 설정
hidden_size = 16

model = LSTMAutoencoder(input_size, hidden_size, timesteps).to(device)
model.eval()

LSTMAutoencoder(
  (encoder_lstm): LSTM(22, 16, batch_first=True)
  (decoder_lstm): LSTM(16, 22, batch_first=True)
)

In [27]:
import torch
import numpy as np
import shap

# 1. 환경 설정 (윈도우 60, 피처 22)
input_size = 22     
timesteps = 60      
device = torch.device('cpu')

# 2. 훈련 데이터에서 가져온 베이스라인 (SHAP 연산의 기준점 역할, 10개 샘플)
# 모델이 정상 상태일 때의 기준을 알기 위해 학습 데이터에서 10개만 추출해 둡니다.
X_train_base = np.random.rand(10, timesteps, input_size).astype(np.float32)
X_train_flattened = X_train_base.reshape(10, -1) # 2차원으로 펼침

# 3. 🎯 바로 '지금 이 순간' 분석하고 싶은 딱 1개의 데이터 (Shape: [60, 22])
# 오류가 발생했거나, 급격한 변화가 일어난 '그 순간'의 60개 타임스텝 데이터입니다.
current_moment_data = np.random.rand(timesteps, input_size).astype(np.float32)

# SHAP에 넣기 위해 배치 차원을 넣고 2차원으로 변환 (Shape: [1, 1320])
current_moment_flattened = current_moment_data.reshape(1, -1)

print(f'#####1')
# 4. '그 순간'의 오차 매트릭을 계산하는 래퍼 함수
def model_moment_error_wrapper(x_flattened):
    # SHAP이 변형하며 테스트하는 데이터를 [Batch, 60, 22]로 복구
    print(f'x_flattened {type(x_flattened)}, {len(x_flattened)}')
    x_3d = x_flattened.reshape(-1, timesteps, input_size)
    x_tensor = torch.tensor(x_3d, dtype=torch.float32).to(device)
    print(f'#####1-1')
    with torch.no_grad():
        outputs = model(x_tensor)
        print(f'#####1-1-1')
        # 🎯 딱 마지막 타임스텝(-1) 순간의 실제값과 예측값의 오차(MAE) 산출
        true_last = x_tensor[:, -1, :]
        pred_last = outputs[:, -1, :]
        print(f'#####1-1-2')
        # 각 배치 샘플별 '마지막 순간의 오차 점수' 1개 도출
        moment_error_metric = torch.mean(torch.abs(true_last - pred_last), dim=1)
        print(f'#####1-1-3')
        print(f'moment_error_metric = {moment_error_metric}')
    print(f'#####1-2')
    return moment_error_metric.cpu().numpy()
print(f'#####2')
print(f'X_train_flattened------------------{type(X_train_flattened)}')
# 5. SHAP Explainer 선언 및 '이 순간의 데이터 1개'만 주입
explainer = shap.KernelExplainer(model_moment_error_wrapper, X_train_flattened)
print(f'explainer------------------{explainer}')
# 🚨 오직 이 순간(current_moment_flattened) 딱 1개만 넣고 영향도를 계산합니다.
print(f'current_moment_flattened------------------{type(current_moment_flattened)}')
print(f'current_moment_flattened------------------{len(current_moment_flattened)}')
print(f'current_moment_flattened------------------{len(current_moment_flattened[0])}')
print(f'current_moment_flattened------------------{current_moment_flattened}')
shap_values_flattened = explainer.shap_values(current_moment_flattened)
print(f'shap_values_flattened------------------{shap_values_flattened}')

# 6. 연산 결과 복원 ([1320] -> [60, 22])
# '이 순간' 입력된 60개 타임스텝 동안 22개 변수가 각각 미친 영향도 지도입니다.
shap_matrix = shap_values_flattened.reshape(timesteps, input_size)

# 7. 최종 분석: 60번의 시간 흐름 동안 22개 피처가 '이 순간 오차'에 준 평균 영향도
feature_importance = np.mean(shap_matrix, axis=0) # 크기: 22
feature_importance = [(idx+1, importance) for idx, importance in enumerate(feature_importance)]

# feature_importance_list = feature_importance.tolist()
# 이 22개 숫자가 바로 '지금 이 순간 오차'를 만들어낸 변수별 지분(영향도)입니다.
print(f'feature_importance = {type(feature_importance)},\n\t\t{feature_importance}')
for i, importance in feature_importance:
    print(f"Feature_{i}의 오차 기여도: {importance:.6f}")
# for i, importance in enumerate(feature_importance):
#     print(f"Feature_{i}의 오차 기여도: {importance:.6f}")
print(f'\n\n{"*"*20}\n\n')
# for i, importance in enumerate(feature_importance_list):
#     print(f"Feature_{i}의 오차 기여도: {importance:.6f}")


#####1
#####2
X_train_flattened------------------<class 'numpy.ndarray'>
x_flattened <class 'numpy.ndarray'>, 10
#####1-1
#####1-1-1
#####1-1-2
#####1-1-3
moment_error_metric = tensor([0.4888, 0.4266, 0.4759, 0.5581, 0.5038, 0.5341, 0.5141, 0.5369, 0.5513,
        0.4959])
#####1-2
explainer------------------<shap.explainers._kernel.KernelExplainer object at 0x79003bc36780>
current_moment_flattened------------------<class 'numpy.ndarray'>
current_moment_flattened------------------1
current_moment_flattened------------------1320
current_moment_flattened------------------[[0.1856998  0.85226476 0.9918189  ... 0.04198371 0.6322016  0.8493053 ]]


  0%|          | 0/1 [00:00<?, ?it/s]

x_flattened <class 'numpy.ndarray'>, 1
#####1-1
#####1-1-1
#####1-1-2
#####1-1-3
moment_error_metric = tensor([0.4736])
#####1-2
x_flattened <class 'numpy.ndarray'>, 46880
#####1-1
#####1-1-1
#####1-1-2
#####1-1-3
moment_error_metric = tensor([0.4888, 0.4266, 0.4759,  ..., 0.4736, 0.4736, 0.4736])
#####1-2


100%|██████████| 1/1 [00:03<00:00,  3.09s/it]

shap_values_flattened------------------[[ 0.          0.          0.         ... -0.02113524  0.
   0.02522547]]
feature_importance = <class 'list'>,
		[(1, np.float64(0.0)), (2, np.float64(0.0)), (3, np.float64(0.0)), (4, np.float64(0.0)), (5, np.float64(0.0)), (6, np.float64(-0.0002393104047031642)), (7, np.float64(0.0)), (8, np.float64(0.0)), (9, np.float64(-0.0002196588128398224)), (10, np.float64(0.0003050234245046928)), (11, np.float64(0.0)), (12, np.float64(0.0002702472786326879)), (13, np.float64(-0.00022194473453556206)), (14, np.float64(-0.00030193305937137603)), (15, np.float64(0.0)), (16, np.float64(0.0)), (17, np.float64(-0.00012993553034349557)), (18, np.float64(-0.00011259093093520036)), (19, np.float64(0.0)), (20, np.float64(-0.0003522540235903575)), (21, np.float64(0.0)), (22, np.float64(0.00042042447695781723))]
Feature_1의 오차 기여도: 0.000000
Feature_2의 오차 기여도: 0.000000
Feature_3의 오차 기여도: 0.000000
Feature_4의 오차 기여도: 0.000000
Feature_5의 오차 기여도: 0.000000
Feature_6의 오차 기여도: